# Agents in LlamaIndex

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

## Let's install the dependencies

We will install the dependencies for this unit.

In [1]:
!pip install llama-index llama-index-vector-stores-chroma llama-index-llms-huggingface-api llama-index-embeddings-huggingface -U -q

And, let's log in to Hugging Face to use serverless Inference APIs.

In [2]:
from huggingface_hub import login

login()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Initialising agents

Let's start by initialising an agent. We will use the basic `AgentWorkflow` class to create an agent.

In [3]:
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.core.agent.workflow import AgentWorkflow, ToolCallResult, AgentStream


def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers"""
    return a - b


def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


def divide(a: int, b: int) -> int:
    """Divide two numbers"""
    return a / b


llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")

agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[subtract, multiply, divide, add],
    llm=llm,
    system_prompt="You are a math agent that can add, subtract, multiply, and divide numbers using provided tools.",
)

Then, we can run the agent and get the response and reasoning behind the tool calls.

In [4]:
handler = agent.run("What is (2 + 2) * 2?")
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp

Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: add
Action Input: {"a": 2, "b": 2}
Called tool:  add {'a': 2, 'b': 2} => 4
Thought: Now I need to multiply the result by 2.
Action: multiply
Action Input: {'a': 4, 'b': 2}
Called tool:  multiply {'a': 4, 'b': 2} => 8
Thought: I can answer without using any more tools. I'll use the user's language to answer
Answer: (2 + 2) * 2 = 8

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='(2 + 2) * 2 = 8')]), tool_calls=[ToolCallResult(tool_name='add', tool_kwargs={'a': 2, 'b': 2}, tool_id='8919ef60-4e45-4100-8a8b-01dd8791ac1d', tool_output=ToolOutput(content='4', tool_name='add', raw_input={'args': (), 'kwargs': {'a': 2, 'b': 2}}, raw_output=4, is_error=False), return_direct=False), ToolCallResult(tool_name='multiply', tool_kwargs={'a': 4, 'b': 2}, tool_id='2cc5cf55-1219-4805-bdfc-65b921481b36', tool_output=ToolOutput(content='8', tool_name='multiply', raw_input={'args': (), 'kwargs': {'a': 4, 'b': 2}}, raw_output=8, is_error=False), return_direct=False)], raw=ChatCompletionStreamOutput(choices=[ChatCompletionStreamOutputChoice(delta=ChatCompletionStreamOutputDelta(role='assistant', content='8', tool_calls=None), index=0, finish_reason=None, logprobs=None)], created=1745002699, id='', model='Qwen/Qwen2.5-Coder-32B-Instruct', syste

In a similar fashion, we can pass state and context to the agent.


In [9]:
resp = await agent.run("My name is Bob.")
# print(resp.content)
print(resp)

await agent.run("What was my name again?")

Hello, Bob! It seems like you used the add tool to add 1 and 2, which equals 3. How can I assist you further?


AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="I don't have enough information to determine your name. Could you please provide it?")]), tool_calls=[ToolCallResult(tool_name='None', tool_kwargs={}, tool_id='440f7be5-50bf-41c1-beb3-a8597c9114d0', tool_output=ToolOutput(content='Tool None not found. Please select a tool that is available.', tool_name='None', raw_input={}, raw_output=None, is_error=True), return_direct=False)], raw=ChatCompletionStreamOutput(choices=[ChatCompletionStreamOutputChoice(delta=ChatCompletionStreamOutputDelta(role='assistant', content='?', tool_calls=None), index=0, finish_reason=None, logprobs=None)], created=1745005039, id='', model='Qwen/Qwen2.5-Coder-32B-Instruct', system_fingerprint='3.2.1-sha-4d28897', usage=None), current_agent_name='Agent')

In [10]:
from llama_index.core.workflow import Context

ctx = Context(agent)

response = await agent.run("My name is Bob.", ctx=ctx)
response = await agent.run("What was my name again?", ctx=ctx)
response

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Your name is Bob.')]), tool_calls=[], raw=ChatCompletionStreamOutput(choices=[ChatCompletionStreamOutputChoice(delta=ChatCompletionStreamOutputDelta(role='assistant', content='.', tool_calls=None), index=0, finish_reason=None, logprobs=None)], created=1745002762, id='', model='Qwen/Qwen2.5-Coder-32B-Instruct', system_fingerprint='3.2.1-sha-4d28897', usage=None), current_agent_name='Agent')

## Creating RAG Agents with QueryEngineTools

Let's now re-use the `QueryEngine` we defined in the [previous unit on tools](/tools.ipynb) and convert it into a `QueryEngineTool`. We will pass it to the `AgentWorkflow` class to create a RAG agent.

### Recreate database

In [2]:
from typing import Sequence


from llama_index.core.schema import BaseNode


import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core import SimpleDirectoryReader


In [ ]:

reader = SimpleDirectoryReader(input_dir="data")
documents = reader.load_data()
print(f"Docs: {len(documents)}")

db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection(name="alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5"),
    ],
    vector_store=vector_store,
)

nodes: Sequence[BaseNode] = await pipeline.arun(documents=documents)

print(f"Nodes: {len(nodes)}")
print(nodes[-1])



In [14]:

from llama_index.core import VectorStoreIndex
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.core.agent.workflow import FunctionAgent, ToolCallResult, ToolCall, AgentStream
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.tools import QueryEngineTool
from llama_index.vector_stores.chroma import ChromaVectorStore


In [40]:

# Create a vector store
db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection("alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Create a query engine
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")


from llama_index.llms.ollama import Ollama
# llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")
model = "llama3.2:1b"
llm = Ollama(model=model, request_timeout=60.0)

from llama_index.llms.openai import OpenAI
llm = OpenAI(model="gpt-4o")



index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, embed_model=embed_model
)
query_engine = index.as_query_engine(llm=llm, similarity_top_k=10)


In [29]:
nodes[-1]

TextNode(id_='d2e2f250-7fbd-43fd-8d73-5a7d9f35be82', embedding=[-0.06982996314764023, -0.016652539372444153, 0.04755483195185661, -0.032662782818078995, 0.03426332399249077, -0.026365717872977257, 0.00815120991319418, 0.06548037379980087, 0.006377530284225941, -0.015981892123818398, 0.04547871649265289, -0.025397546589374542, 0.03486974537372589, 0.0003721399698406458, 0.006271654739975929, 0.031328435987234116, 0.007850478403270245, 0.032547544687986374, -0.004539307672530413, -0.0001433026191079989, 0.06861965358257294, -0.007620756048709154, 0.055872056633234024, -0.06623724848031998, 0.0257248617708683, 0.005440917797386646, -0.057903602719306946, -0.06415807455778122, -0.03451094403862953, -0.10629883408546448, -0.01484108716249466, 0.00803904514759779, 0.03306661173701286, 0.05878249928355217, -0.019034262746572495, 0.014371546916663647, 0.01622423157095909, 0.001532128662802279, -0.036256104707717896, 0.027380049228668213, 0.051389556378126144, -0.011143666692078114, -0.03063197

In [41]:
for node in nodes:
    print(node.metadata["file_name"], node.text) if  "mathematician" in node.text else ""

persona_1398.txt A mathematician specializing in Euclidean geometry, particularly triangle geometry.
persona_2201.txt A mathematician specializing in topology, likely an academic or researcher focused on geometric and topological properties of spaces.
persona_2207.txt A mathematician trained in geometric construction proofs, likely specializing in number theory or algebraic geometry.
persona_266.txt A mathematician with an interest in algebra and number theory, likely an academic researcher or educator, with a focus on abstract and theoretical concepts.
persona_2709.txt An academic or research mathematician specializing in the history of mathematics.
persona_2732.txt A university mathematics professor with a specialization in the history of mathematics, or a mathematician with a strong interest in the historical development of mathematical theories.
persona_32.txt A mathematician, likely with a background in the history and philosophy of mathematics, and possibly a university professor

In [19]:
nodes[-1]

TextNode(id_='a41448e6-184d-47f1-a477-b80f4a01d21f', embedding=[-0.06982996314764023, -0.016652539372444153, 0.04755483195185661, -0.032662782818078995, 0.03426332399249077, -0.026365717872977257, 0.00815120991319418, 0.06548037379980087, 0.006377530284225941, -0.015981892123818398, 0.04547871649265289, -0.025397546589374542, 0.03486974537372589, 0.0003721399698406458, 0.006271654739975929, 0.031328435987234116, 0.007850478403270245, 0.032547544687986374, -0.004539307672530413, -0.0001433026191079989, 0.06861965358257294, -0.007620756048709154, 0.055872056633234024, -0.06623724848031998, 0.0257248617708683, 0.005440917797386646, -0.057903602719306946, -0.06415807455778122, -0.03451094403862953, -0.10629883408546448, -0.01484108716249466, 0.00803904514759779, 0.03306661173701286, 0.05878249928355217, -0.019034262746572495, 0.014371546916663647, 0.01622423157095909, 0.001532128662802279, -0.036256104707717896, 0.027380049228668213, 0.051389556378126144, -0.011143666692078114, -0.03063197

In [43]:
response = query_engine.query(
    "Respond using a persona that describes someone related with mathematics"
)

print(response)

As a mathematics educator focused on geometry, I strive to provide clear and detailed explanations to help students understand complex concepts. Whether teaching at a high school or introductory college level, my goal is to make geometry accessible and engaging through a question-and-answer format that encourages active learning and curiosity.


In [44]:

query_engine_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="personas",
    description="descriptions for various types of personas",
    return_direct=False,
)

# Create a RAG agent
query_engine_agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[query_engine_tool],
    llm=llm,
    system_prompt="You are a helpful assistant that has access to a database containing persona descriptions. ",
)

And, we can once more get the response and reasoning behind the tool calls.

In [27]:
handler = query_engine_agent.run(
    "Search the database for 'science fiction' and return some persona descriptions."
)
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp


Called tool:  personas {'input': 'science fiction'} => The context does not provide specific information about science fiction. It focuses on an astronomy enthusiast or a science journalist interested in space discoveries and cosmological research.
It seems that the database doesn't have specific persona descriptions labeled under "science fiction." However, it does mention personas related to astronomy enthusiasts or science journalists interested in space discoveries and cosmological research. If you need more detailed information or a different type of persona, feel free to ask!

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='It seems that the database doesn\'t have specific persona descriptions labeled under "science fiction." However, it does mention personas related to astronomy enthusiasts or science journalists interested in space discoveries and cosmological research. If you need more detailed information or a different type of persona, feel free to ask!')]), tool_calls=[ToolCallResult(tool_name='personas', tool_kwargs={'input': 'science fiction'}, tool_id='call_utsd3uvhx3cz19cCbjh8bpFT', tool_output=ToolOutput(content='The context does not provide specific information about science fiction. It focuses on an astronomy enthusiast or a science journalist interested in space discoveries and cosmological research.', tool_name='personas', raw_input={'input': 'science fiction'}, raw_output=Response(response='The context does not provide specific information about scien

In [28]:
import pprint

for block in resp.response.blocks:
    pprint.pprint(block.text) if block.block_type == "text" else ""

("It seems that the database doesn't have specific persona descriptions "
 'labeled under "science fiction." However, it does mention personas related '
 'to astronomy enthusiasts or science journalists interested in space '
 'discoveries and cosmological research. If you need more detailed information '
 'or a different type of persona, feel free to ask!')


## Creating multi-agent systems

We can also create multi-agent systems by passing multiple agents to the `AgentWorkflow` class.

In [45]:
from llama_index.core.agent.workflow import (
    AgentWorkflow,
    ReActAgent,
)


# Define some tools
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers."""
    return a - b


# Create agent configs
# NOTE: we can use FunctionAgent or ReActAgent here.
# FunctionAgent works for LLMs with a function calling API.
# ReActAgent works for any LLM.
calculator_agent = ReActAgent(
    name="calculator",
    description="Performs basic arithmetic operations",
    system_prompt="You are a calculator assistant. Use your tools for any math operation.",
    tools=[add, subtract],
    llm=llm,
)

query_agent = ReActAgent(
    name="info_lookup",
    description="Looks up information about different professions or personas",
    system_prompt="Use your tool to query a RAG system to answer information about different professions or personas",
    tools=[query_engine_tool],
    llm=llm,
)

# Create and run the workflow
agent = AgentWorkflow(agents=[calculator_agent, query_agent], root_agent="calculator")

# Run the system
handler = agent.run(user_msg="Can you add 5 and 3?")

In [31]:
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp

Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: add
Action Input: {"a": 5, "b": 3}
Called tool:  add {'a': 5, 'b': 3} => 8
Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: The sum of 5 and 3 is 8.

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='The sum of 5 and 3 is 8.')]), tool_calls=[ToolCallResult(tool_name='add', tool_kwargs={'a': 5, 'b': 3}, tool_id='32355b30-5223-4ab2-8a81-619e24b8ac89', tool_output=ToolOutput(content='8', tool_name='add', raw_input={'args': (), 'kwargs': {'a': 5, 'b': 3}}, raw_output=8, is_error=False), return_direct=False)], raw={'id': 'chatcmpl-BNor3UW0kiwVPrkQYD1JjP2VeoxJz', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None}], 'created': 1745016729, 'model': 'gpt-4o-2024-08-06', 'object': 'chat.completion.chunk', 'service_tier': 'default', 'system_fingerprint': 'fp_f5bdcc3276', 'usage': None}, current_agent_name='calculator')

In [46]:
# Run the system
handler = agent.run(user_msg="Can you add 5 and 3? Then summarize itemized personas describing a matematitian")
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp

Thought: The current language of the user is English. I need to use a tool to add the numbers 5 and 3.
Action: add
Action Input: {"a": 5, "b": 3}
Called tool:  add {'a': 5, 'b': 3} => 8
Thought: I have successfully added the numbers. Now, I need to summarize itemized personas describing a mathematician. Since I don't have the capability to summarize personas, I will hand off to the appropriate agent.
Action: handoff
Action Input: {"to_agent": "info_lookup", "reason": "To summarize itemized personas describing a mathematician."}
Called tool:  handoff {'to_agent': 'info_lookup', 'reason': 'To summarize itemized personas describing a mathematician.'} => Agent info_lookup is now handling the request due to the following reason: To summarize itemized personas describing a mathematician..
Please continue with the current request.
Thought: I need to use the personas tool to get descriptions for a mathematician and then summarize them.
Action: personas
Action Input: {"input": "mathematician"}


AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='A mathematician is typically an academic or researcher who may specialize in areas such as algebra, number theory, abstract and theoretical concepts, the history of mathematics, topology, or geometric construction proofs. Their work involves in-depth exploration of specific mathematical theories and properties.')]), tool_calls=[ToolCallResult(tool_name='add', tool_kwargs={'a': 5, 'b': 3}, tool_id='5741ce74-08be-4bbe-be42-7f19a16de8ed', tool_output=ToolOutput(content='8', tool_name='add', raw_input={'args': (), 'kwargs': {'a': 5, 'b': 3}}, raw_output=8, is_error=False), return_direct=False), ToolCallResult(tool_name='handoff', tool_kwargs={'to_agent': 'info_lookup', 'reason': 'To summarize itemized personas describing a mathematician.'}, tool_id='584a275a-d55a-4bbe-ab78-bf51171b92ee', tool_output=ToolOutput(content='Agent info_lookup is now handlin

In [49]:
# Run the system
handler = agent.run(user_msg="Can a mathematitian explain Markov chains? Use the available tool for personas")
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp


Thought: The current language of the user is English. I need to use a tool to hand off this request to the appropriate agent who can provide information about a mathematician explaining Markov chains.
Action: handoff
Action Input: {"to_agent": "info_lookup", "reason": "Looks up information about different professions or personas, such as a mathematician explaining Markov chains."}
Called tool:  handoff {'to_agent': 'info_lookup', 'reason': 'Looks up information about different professions or personas, such as a mathematician explaining Markov chains.'} => Agent info_lookup is now handling the request due to the following reason: Looks up information about different professions or personas, such as a mathematician explaining Markov chains..
Please continue with the current request.
Thought: I need to use the personas tool to get a description of a mathematician explaining Markov chains.
Action: personas
Action Input: {"input": "mathematician explaining Markov chains"}
Called tool:  pers

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='A mathematician with expertise in mathematical modeling, particularly in areas like disease spread and infectious diseases, would be well-suited to explain Markov chains. They possess a deep understanding of complex mathematical concepts and have the ability to convey these ideas in a way that is accessible and understandable to a general audience. Markov chains are mathematical systems that undergo transitions from one state to another on a state space, and such a mathematician can effectively explain how these transitions work and their applications.')]), tool_calls=[ToolCallResult(tool_name='handoff', tool_kwargs={'to_agent': 'info_lookup', 'reason': 'Looks up information about different professions or personas, such as a mathematician explaining Markov chains.'}, tool_id='3a05b0a3-167c-4d41-b5c7-d1aee964d5e0', tool_output=ToolOutput(content='A